# 04 - Run and poll a job

Kick off a B2C job execution and wait for it to reach a terminal status. The
OCAPI endpoints and the Account Manager token endpoint are mocked with `respx`,
and the poll loop's sleep is replaced with a no-op so it returns instantly.

Public API: `B2CInstance`, `execute_job`, `wait_for_job`, `WaitForJobOptions`.

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

In [ ]:
# The SDK decodes (does NOT verify) Account Manager access tokens, so a valid
# JWT *shape* is enough for a mocked token endpoint. This mirrors the unsigned
# test JWTs used by the SDK's own suite (tests/helpers/jwt.py).
def make_jwt(*, expires_in=3600, scope=None, sub=None):
    def b64(raw: bytes) -> str:
        return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")

    header = {"alg": "RS256", "typ": "JWT"}
    payload = {"exp": int(time.time()) + expires_in}
    if scope is not None:
        payload["scope"] = scope
    if sub is not None:
        payload["sub"] = sub
    return ".".join([b64(json.dumps(header).encode()), b64(json.dumps(payload).encode()), "sig"])

## Build a `B2CInstance`

A hostname plus OAuth client-credentials is all OCAPI needs. The instance builds
its OCAPI client and OAuth strategy lazily.

In [ ]:
from b2c_tooling_sdk import (
    DEFAULT_ACCOUNT_MANAGER_HOST,
    AuthConfig,
    B2CInstance,
    InstanceConfig,
    execute_job,
    wait_for_job,
)
from b2c_tooling_sdk.auth import OAuthAuthConfig
from b2c_tooling_sdk.clients.ocapi import DEFAULT_API_VERSION
from b2c_tooling_sdk.operations.jobs import WaitForJobOptions

HOSTNAME = "example.demandware.net"
JOB_ID = "my-export-job"

instance = B2CInstance(
    InstanceConfig(hostname=HOSTNAME),
    AuthConfig(oauth=OAuthAuthConfig(client_id="cid", client_secret="secret")),
)

TOKEN_URL = f"https://{DEFAULT_ACCOUNT_MANAGER_HOST}/dwsso/oauth2/access_token"
OCAPI_BASE = f"https://{HOSTNAME}/s/-/dw/data/{DEFAULT_API_VERSION}"
print("OCAPI base URL:", OCAPI_BASE)

## Execute the job, then poll to completion

`execute_job` POSTs a new execution; `wait_for_job` polls until the execution is
`finished` (or errors / times out). We inject `sleep=fast_sleep` so no real time
passes, and record each poll via `on_poll`.

In [ ]:
async def fast_sleep(_seconds: float) -> None:
    """A no-op sleep so the poll loop returns instantly."""
    return None


polls: list[str] = []

with respx.mock(assert_all_called=False) as router:
    # Account Manager token endpoint (used by the OCAPI OAuth strategy).
    router.post(TOKEN_URL).mock(
        return_value=httpx.Response(
            200,
            json={"access_token": make_jwt(scope="sfcc.jobs"), "expires_in": 1800, "scope": "sfcc.jobs"},
        )
    )
    # Start the execution.
    router.post(f"{OCAPI_BASE}/jobs/{JOB_ID}/executions").mock(
        return_value=httpx.Response(200, json={"id": "exec-42", "execution_status": "pending"})
    )
    # First poll reports a terminal, successful status.
    router.get(f"{OCAPI_BASE}/jobs/{JOB_ID}/executions/exec-42").mock(
        return_value=httpx.Response(
            200,
            json={"id": "exec-42", "execution_status": "finished", "exit_status": {"code": "OK"}},
        )
    )

    started = await execute_job(instance, JOB_ID)
    print("Started execution:", started["id"], "->", started["execution_status"])

    final = await wait_for_job(
        instance,
        JOB_ID,
        started["id"],
        WaitForJobOptions(sleep=fast_sleep, on_poll=lambda info: polls.append(info.status)),
    )

print("Poll statuses:", polls)
print("Terminal status:", final["execution_status"], final["exit_status"]["code"])
assert final["execution_status"] == "finished"

## Recap

- `execute_job` started an execution over OCAPI.
- `wait_for_job` polled to a terminal `finished`/`OK` status, using an injected
  no-op sleep so the example is instant and deterministic.